# L15a Example: Hebbian associative memory in the rate domain
In this example, we build the Hebbian Memory module of [Limbacher and Legenstein (NeurIPS 2020)](https://proceedings.neurips.cc/paper/2020/file/f6876a9f998f6472cc26708e27444456-Paper.pdf) in isolation, with no LIF dynamics and no encoder training. We generate $K$ random sparse key-value pairs in $\mathbb{R}^{m}$, write them into a fresh association matrix $\mathbf{W}^{\text{assoc}}\in\mathbb{R}^{m\times m}$ via the Hebbian outer-product rule, and read each one back with the matrix-vector product $\mathbf{v} = \mathbf{W}^{\text{assoc}}\,\mathbf{k}$. We then characterize the module along two axes that decide whether it is useful in practice: the load factor $K/m$ (how many associations can we cram into a memory of size $m$ before recall breaks down?) and the query corruption $p$ (how robust is recall to a partial cue?).

* __Problem statement.__ Let $\{(\mathbf{k}^{(i)}, \mathbf{v}^{(i)})\}_{i=1}^{K}\subset\mathbb{R}^{m}\times\mathbb{R}^{m}$ be a set of sparse non-negative key-value pairs (the regime that the ReLU encoders in H-Mem produce). We initialize $\mathbf{W}^{\text{assoc}} = \mathbf{0}$, present the pairs sequentially, and apply the Hebbian write rule $\Delta W^{\text{assoc}}_{kj} = \gamma_{+}(w^{\max} - W^{\text{assoc}}_{kj})\,v^{(i)}_{k}\,k^{(i)}_{j} - \gamma_{-}\,W^{\text{assoc}}_{kj}\,(k^{(i)}_{j})^{2}$ at each presentation. We then probe the matrix with each clean key $\mathbf{k}^{(i)}$ and (later) with corrupted variants of it, and measure the Pearson correlation between the recalled $\hat{\mathbf{v}}^{(i)} = \mathbf{W}^{\text{assoc}}\,\mathbf{k}^{(i)}$ and the true $\mathbf{v}^{(i)}$.

> __Learning Objectives.__ After working through this notebook you should be able to:
>
> * __Implement the Hebbian write and read rules of the H-Mem memory module:__ Apply the outer-product update with the Oja-style forgetting term to a fresh association matrix as $K$ key-value pairs are presented, and recover the associated value of any stored key by a single matrix-vector product.
> * __Reason about hetero-associative memory capacity in terms of the load factor $K/m$:__ Sweep the number of stored pairs at fixed memory size, identify the regime where recall correlation stays close to one, the regime where it degrades gracefully, and the load factor at which the knee occurs.
> * __Explain why Hebbian recall is robust to partial cues:__ Corrupt a controlled fraction of each query key's entries, observe that recall correlation falls smoothly (not catastrophically) as corruption grows, and connect this to the matrix-vector form of the read rule.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, which sets paths, loads external packages, and pulls in the H-Mem source files from `src/`:
* [`src/Types.jl`](src/Types.jl) defines the `MyHMemMemory` struct that holds the association matrix and the constants of the write rule.
* [`src/Factory.jl`](src/Factory.jl) defines the `build(MyHMemMemory; ...)` constructor.
* [`src/Compute.jl`](src/Compute.jl) defines `random_sparse_pair`, `hebbian_write!`, `hebbian_read`, and `corrupt_key`.

In [ ]:
include("Include.jl"); # load packages, src/ files, set random seed

### Constants
We fix the memory dimension, the write-rule constants, and the sweep grids up front so every cell below reads from a single set of knobs. The defaults for $\gamma_{+}, \gamma_{-}, w^{\max}$ follow Table 2 of [Limbacher, Özdenizci, and Legenstein (2022)](https://arxiv.org/abs/2205.11276), the spiking-version paper that inherits these constants from the rate-version paper we are reproducing here.

In [ ]:
m            = 64                         # dimension of the key and value vectors
γ_plus       = 0.3                        # Hebbian "fire together, wire together" gain (Limbacher 2022, Tab. 2)
γ_minus      = 0.3                        # Oja-style forgetting gain (Limbacher 2022, Tab. 2)
w_max        = 1.0                        # soft upper bound on entries of W (Limbacher 2022, Tab. 2)
SPARSITY     = 0.1                        # probability that any individual entry of k or v is non-zero
K_BASELINE   = 8                          # number of (k, v) pairs to write in Task 1 (load factor K/m = 0.125)
K_SWEEP      = [1, 2, 4, 8, 16, 32, 64, 96, 128]    # number of stored pairs to sweep over in Task 2
P_SWEEP      = [0.0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]   # query corruption fractions to sweep in Task 3
K_NOISE      = 8                          # number of stored pairs in the noise sweep (load factor 0.125)
N_REPLICATES = 16;                        # independent random restarts averaged over in each sweep cell

___

## Task 1: Write and read $K$ key-value pairs
In this task, we instantiate a fresh `MyHMemMemory` with association matrix $\mathbf{W}^{\text{assoc}} = \mathbf{0}$, present $K =$ `K_BASELINE` key-value pairs sequentially, and verify that each can be read back from the resulting matrix.

> __Write rule and read rule.__
>
> The Hebbian write rule from Eq. 1 of [Limbacher and Legenstein (2020)](https://proceedings.neurips.cc/paper/2020/file/f6876a9f998f6472cc26708e27444456-Paper.pdf) updates each entry of $\mathbf{W}^{\text{assoc}}$ by $\Delta W^{\text{assoc}}_{kj} = \gamma_{+}(w^{\max} - W^{\text{assoc}}_{kj})\,v_{k}\,k_{j} - \gamma_{-}\,W^{\text{assoc}}_{kj}\,k_{j}^{2}$, with the soft upper bound $w^{\max}$ keeping entries from saturating and the Oja-style second term weakening connections from currently active key-coordinates so that fresh associations overwrite stale ones. The read rule is the matrix-vector product $\hat{\mathbf{v}} = \mathbf{W}^{\text{assoc}}\,\mathbf{k}^{q}$ from Eq. 5 of the same paper.

The code block below stores the freshly written memory in `mem::MyHMemMemory`, the stored keys and values in `keys_store::Vector{Vector{Float64}}` and `vals_store::Vector{Vector{Float64}}`, and the per-pair recall correlations in `ρ_store::Vector{Float64}`.

In [ ]:
mem, keys_store, vals_store, ρ_store = let
    # Re-seed locally so the example is reproducible regardless of any earlier
    # random draws in the notebook.
    Random.seed!(7)

    # Step 1: instantiate a fresh memory with W = 0.
    mem = build(MyHMemMemory; m = m, γ_plus = γ_plus, γ_minus = γ_minus, w_max = w_max)

    # Step 2: draw K_BASELINE random sparse (k, v) pairs and write each one
    # into the association matrix via the Hebbian outer-product rule.
    keys_store = Vector{Vector{Float64}}(undef, K_BASELINE)
    vals_store = Vector{Vector{Float64}}(undef, K_BASELINE)
    for i in 1:K_BASELINE
        k, v = random_sparse_pair(m; sparsity = SPARSITY)
        keys_store[i] = k
        vals_store[i] = v
        hebbian_write!(mem, k, v)
    end

    # Step 3: read each clean key back and record the per-pair recall correlation.
    ρ_store = [cor(hebbian_read(mem, keys_store[i]), vals_store[i]) for i in 1:K_BASELINE]

    @info "Hebbian write & read" m K=K_BASELINE load=round(K_BASELINE / m; digits = 3) mean_ρ=round(mean(ρ_store); digits = 3) min_ρ=round(minimum(ρ_store); digits = 3) max_W=round(maximum(mem.W); digits = 3)
    (mem, keys_store, vals_store, ρ_store)
end;

### Visualize one read: $\hat{\mathbf{v}}$ vs. $\mathbf{v}$
We pick the first stored pair and overlay the recalled value vector $\hat{\mathbf{v}}^{(1)} = \mathbf{W}^{\text{assoc}}\,\mathbf{k}^{(1)}$ against the stored value vector $\mathbf{v}^{(1)}$. A perfect read sits exactly on the identity line; mild interference from the other $K-1$ pairs shows up as scatter.

In [ ]:
let
    # Pick the first stored pair and read it back.
    v_hat = hebbian_read(mem, keys_store[1])
    v_true = vals_store[1]
    ρ = round(cor(v_hat, v_true); digits = 3)

    # Identity line spans the joint range so the perfect-read locus is easy to see.
    lo = min(minimum(v_hat), minimum(v_true))
    hi = max(maximum(v_hat), maximum(v_true))

    p = scatter(v_true, v_hat;
        ms = 4, msw = 0, c = :deepskyblue,
        label = "recalled vs true (ρ = $(ρ))",
        xlabel = "true value entry vᵢ", ylabel = "recalled value entry v̂ᵢ",
        title  = "Hebbian read of stored pair 1 (m = $(m), K = $(K_BASELINE))",
        bg = "gray95", background_color_outside = "white",
        framestyle = :box, fg_legend = :transparent)
    plot!(p, [lo, hi], [lo, hi]; c = :red, ls = :dash, lw = 1.2, label = "identity")

    plot(p; size = (700, 500),
        left_margin = 12Plots.mm, bottom_margin = 10Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

### Visualize $\mathbf{W}^{\text{assoc}}$ after $K$ writes
We render the association matrix as a heatmap. Because each write is an outer product $\mathbf{v}^{(i)}\mathbf{k}^{(i)\top}$ on top of the Oja-style decay, the matrix becomes a sparse low-rank object whose non-zero structure reflects which (key-coord, value-coord) pairs co-fired during the writes.

In [ ]:
let
    # Heatmap of W. Rows are post-synaptic value-coordinates, columns are
    # pre-synaptic key-coordinates. Sparse pairs leave a sparse W behind.
    p = heatmap(mem.W;
        c = :viridis, clims = (0.0, w_max),
        xlabel = "pre-synaptic key index j",
        ylabel = "post-synaptic value index k",
        title  = "W^assoc after $(K_BASELINE) Hebbian writes (m = $(m))",
        yflip = true,
        bg = "gray95", background_color_outside = "white",
        framestyle = :box)

    plot(p; size = (700, 600),
        left_margin = 12Plots.mm, bottom_margin = 10Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

### Things to think about
* __What does the heatmap tell you about which pairs are stored?__ Using only the heatmap and the sparsity (`SPARSITY = 0.1`) of the keys and values: (a) Roughly what fraction of the $m^{2}$ entries of $\mathbf{W}^{\text{assoc}}$ are non-zero, and how does this fraction relate to the per-vector sparsity squared? (b) If you doubled `K_BASELINE`, would you expect the heatmap to become more uniformly populated or to keep its current low-density structure, and why does the soft upper bound $w^{\max}$ cap how dark a single entry can become?

___

## Task 2: Capacity sweep — recall correlation vs. load factor $K/m$
In this task, we vary the number of stored pairs $K\in$ `K_SWEEP` at fixed memory size $m$ and measure how recall correlation changes. The load factor $K/m$ is the natural axis: it tells us how many associations the memory is being asked to keep simultaneously, normalized by its dimension.

> __What do we expect?__
>
> When $K/m\ll 1$ the matrix has plenty of headroom and recall correlation should be close to one. As $K/m$ approaches and then exceeds one, every additional pair starts overwriting earlier ones via the Oja-style forgetting term, and recall correlation falls. The decline should be _graceful_ rather than catastrophic, because the read rule is a linear projection that averages contributions from all stored pairs; this is the same robustness property that makes Hopfield-style associative networks degrade gracefully past their capacity limit.

We average over `N_REPLICATES` independent random restarts at each $K$ and report the mean and standard deviation of the recall correlation across replicates and across the $K$ stored pairs in each replicate. The code block below stores the sweep results in `sweep_K::DataFrame`.

In [ ]:
sweep_K = let
    df = DataFrame(K = Int[], load = Float64[], ρ_mean = Float64[], ρ_std = Float64[])

    for K in K_SWEEP
        # Collect one mean recall correlation per replicate, then aggregate.
        ρ_per_rep = Float64[]
        for rep in 1:N_REPLICATES
            # Reseed deterministically so every (K, rep) pair is reproducible.
            Random.seed!(1000 * K + rep)
            mem_rep = build(MyHMemMemory; m = m, γ_plus = γ_plus, γ_minus = γ_minus, w_max = w_max)

            # Generate K (k, v) pairs and write them all in.
            ks = Vector{Vector{Float64}}(undef, K)
            vs = Vector{Vector{Float64}}(undef, K)
            for i in 1:K
                ks[i], vs[i] = random_sparse_pair(m; sparsity = SPARSITY)
                hebbian_write!(mem_rep, ks[i], vs[i])
            end

            # Read each clean key back and average the per-pair correlations
            # for this replicate. NaNs (zero-norm vectors) are filtered out.
            ρs = Float64[]
            for i in 1:K
                v_hat = hebbian_read(mem_rep, ks[i])
                if std(v_hat) > 0 && std(vs[i]) > 0
                    push!(ρs, cor(v_hat, vs[i]))
                end
            end
            push!(ρ_per_rep, isempty(ρs) ? 0.0 : mean(ρs))
        end
        push!(df, (K, K / m, mean(ρ_per_rep), std(ρ_per_rep)))
    end

    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact),
        formatters = [(v, i, j) -> v isa AbstractFloat ? round(v; sigdigits = 4) : v])
    df
end;

In [ ]:
let
    # Plot recall correlation vs. load factor K/m on a log-2 x-axis because
    # K_SWEEP doubles geometrically up to m, then continues into the overload regime.
    p = plot(sweep_K.load, sweep_K.ρ_mean;
        ribbon = sweep_K.ρ_std, fillalpha = 0.2,
        label = "mean recall correlation", lw = 2, marker = :circle, c = :deepskyblue,
        xlabel = "load factor K / m",
        ylabel = "recall correlation ρ",
        title  = "H-Mem capacity sweep (m = $(m), $(N_REPLICATES) replicates)",
        xscale = :log2,
        ylims = (-0.1, 1.05),
        legend = :bottomleft,
        bg = "gray95", background_color_outside = "white",
        framestyle = :box, fg_legend = :transparent)
    vline!(p, [1.0]; c = :red, ls = :dash, lw = 1.2, label = "K = m")

    plot(p; size = (900, 460),
        left_margin = 12Plots.mm, bottom_margin = 10Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

### Things to think about
* __What does the sweep table tell you about capacity?__ Using only the table and the plot: (a) At what load factor does the recall correlation first fall below $0.9$, and where is the knee that separates the high-fidelity regime from the graceful-degradation regime? (b) Does the standard deviation across replicates grow, shrink, or stay flat as $K/m$ increases past one, and what does that say about how predictable a single random instantiation of the memory becomes once it is overloaded?

___

## Task 3: Noisy-key sweep — recall correlation vs. query corruption $p$
In this task, we hold the load factor fixed at $K =$ `K_NOISE` (a comfortable load of $K/m =$ 0.125) and corrupt a fraction $p\in$ `P_SWEEP` of each query key's entries before reading. Each corrupted entry is replaced by an independent `Uniform(0, 1)` sample; the rest of the entries pass through unchanged. The recall is then $\hat{\mathbf{v}} = \mathbf{W}^{\text{assoc}}\,\tilde{\mathbf{k}}$ where $\tilde{\mathbf{k}}$ is the corrupted query.

> __What do we expect?__
>
> The read is a linear function of the query, so corrupting a fraction $p$ of the query entries should remove roughly that fraction of the signal in the read. Recall correlation should therefore fall smoothly with $p$ rather than collapse all at once: this is the partial-cue robustness that makes Hebbian memory useful when the query is incomplete or noisy. At $p = 1$ the query is independent of the stored key and recall correlation should average to zero.

We again average over `N_REPLICATES` random restarts. The code block below stores the sweep in `sweep_p::DataFrame`.

In [ ]:
sweep_p = let
    df = DataFrame(p = Float64[], ρ_mean = Float64[], ρ_std = Float64[])

    for p in P_SWEEP
        ρ_per_rep = Float64[]
        for rep in 1:N_REPLICATES
            # Distinct seed per (p, rep) for reproducibility. The integer cast on
            # round(Int, 1000 * p) keeps the seed an integer for any p in P_SWEEP.
            Random.seed!(round(Int, 10_000 * p) + rep)
            mem_rep = build(MyHMemMemory; m = m, γ_plus = γ_plus, γ_minus = γ_minus, w_max = w_max)

            # Write K_NOISE clean (k, v) pairs.
            ks = Vector{Vector{Float64}}(undef, K_NOISE)
            vs = Vector{Vector{Float64}}(undef, K_NOISE)
            for i in 1:K_NOISE
                ks[i], vs[i] = random_sparse_pair(m; sparsity = SPARSITY)
                hebbian_write!(mem_rep, ks[i], vs[i])
            end

            # Read each pair back with a CORRUPTED query at level p.
            ρs = Float64[]
            for i in 1:K_NOISE
                k_noisy = corrupt_key(ks[i], p)
                v_hat   = hebbian_read(mem_rep, k_noisy)
                if std(v_hat) > 0 && std(vs[i]) > 0
                    push!(ρs, cor(v_hat, vs[i]))
                end
            end
            push!(ρ_per_rep, isempty(ρs) ? 0.0 : mean(ρs))
        end
        push!(df, (p, mean(ρ_per_rep), std(ρ_per_rep)))
    end

    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact),
        formatters = [(v, i, j) -> v isa AbstractFloat ? round(v; sigdigits = 4) : v])
    df
end;

In [ ]:
let
    # Plot recall correlation vs. query corruption. Linear x-axis is fine here
    # because P_SWEEP is on a unit interval rather than geometric.
    p = plot(sweep_p.p, sweep_p.ρ_mean;
        ribbon = sweep_p.ρ_std, fillalpha = 0.2,
        label = "mean recall correlation", lw = 2, marker = :circle, c = :darkorange,
        xlabel = "query corruption fraction p",
        ylabel = "recall correlation ρ",
        title  = "H-Mem partial-cue robustness (m = $(m), K = $(K_NOISE), $(N_REPLICATES) replicates)",
        ylims  = (-0.2, 1.05),
        legend = :bottomleft,
        bg = "gray95", background_color_outside = "white",
        framestyle = :box, fg_legend = :transparent)
    hline!(p, [0.0]; c = :gray, ls = :dot, lw = 1.0, label = "chance (ρ = 0)")

    plot(p; size = (900, 460),
        left_margin = 12Plots.mm, bottom_margin = 10Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

### Things to think about
* __What does the noise sweep tell you about read robustness?__ Using only the table and the plot: (a) Is the rate at which $\rho$ falls between $p = 0$ and $p = 0.5$ roughly linear, or is the curve concave/convex? Connect this to the fact that the read is a linear function of the query. (b) At $p = 1$ the query has been entirely replaced by random noise, so the signal in the recall comes only from whatever overlap a fresh random vector has with the stored keys — what does the observed ρ at $p = 1$ tell you about that overlap, and how would it change if you raised the per-vector sparsity from `SPARSITY = 0.1` toward `SPARSITY = 0.5`?

___

## Summary
This notebook implemented the rate-domain Hebbian Memory module of [Limbacher and Legenstein (2020)](https://proceedings.neurips.cc/paper/2020/file/f6876a9f998f6472cc26708e27444456-Paper.pdf) in isolation: a single association matrix $\mathbf{W}^{\text{assoc}}\in\mathbb{R}^{m\times m}$ written by an outer-product Hebbian rule and read by a matrix-vector product. We characterized its capacity by sweeping the load factor $K/m$ and its robustness by sweeping the corruption fraction $p$ of the query key. No encoder was trained; everything we observed reflects the dynamics of the write and read rules on a freshly initialized memory.

> __Key Takeaways:__
>
> * **Hebbian write plus matrix-vector read implements one-shot hetero-associative memory:** Each write is one outer-product update applied on top of a soft upper bound and an Oja-style forgetting term, so memorizing a new (key, value) pair takes exactly one forward pass with no gradient descent and no parameter updates. The read is a single matrix-vector product against the same matrix.
> * **Capacity scales with the memory dimension $m$, with a graceful-degradation knee near $K = m$:** Recall correlation stays close to one while the load factor $K/m$ is small and falls smoothly past $K/m \approx 1$ as the Oja-style term overwrites earlier associations. The standard deviation across random restarts grows in the overload regime, reflecting that a single instantiation of the memory becomes more sensitive to its particular sequence of writes once it is overloaded.
> * **Recall is robust to partial cues because the read is linear in the query:** Corrupting a fraction $p$ of the query entries removes roughly that fraction of signal from the read, so the recall correlation falls smoothly with $p$ rather than collapsing all at once. This is the property that makes Hebbian memory useful as a building block when the query encoder is imperfect or the cue is incomplete.

Taken together, the two sweeps show why the H-Mem memory module is attractive as the long-time-scale state of a network whose neurons otherwise only remember the last $\tau_{m}\approx 20$ ms of input. The L15b lab puts this same module back on top of LIF neurons with spike-trace plasticity and applies it to one-shot recall of MNIST images.
___